# Lesson 6.1 · The principal components

PCA answers one question: **along which directions do the data vary the most?** It finds a new system of axes (the principal components), ordered by the variance they capture, and lets you
keep only the first ones. It is not a prediction model: it is a **transformation of the data** used to reduce dimensions, visualize, remove noise or decorrelate the features.

**What you will learn**

- What PCA computes (centering, SVD) and how to read `components_`, `explained_variance_ratio_` and the scores.
- Why the scale of the columns decides the result.
- How to reconstruct the data from the components.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
# two correlated variables: a cloud stretched along a diagonal
x1 = rng.normal(0, 2, 300)
X2d = np.c_[x1, 0.8 * x1 + rng.normal(0, 0.7, 300)]
pca2 = PCA().fit(X2d)
print("explained variance per component:", pca2.explained_variance_ratio_.round(3))
print("first component (direction):", pca2.components_[0].round(3))

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(*X2d.T, s=8, alpha=0.4)
for comp, var in zip(pca2.components_, pca2.explained_variance_):
    ax.arrow(*pca2.mean_, *(comp * 2 * np.sqrt(var)), color="red", width=0.04, head_width=0.25)
ax.set_aspect("equal"); plt.show()

## How it works

1. **Center** every column (subtract the mean): PCA studies the variance around the center of the cloud.
2. Find the direction along which the data have **maximum variance**: it is the first component. The second is the direction of maximum variance **orthogonal to the first**, and so on.
3. The result is a rotation: the rotated points (the **scores**) have columns that are uncorrelated with each other, ordered by decreasing variance.

Numerically it is obtained with the **SVD** of the centered matrix $X_c = U\,S\,V^\top$: the rows of $V^\top$ are the components (`components_`), the variances are $S^2/(n-1)$.

In [ ]:
X = load_wine(as_frame=True).data
Xc = (X - X.mean()).to_numpy()
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
variances = S ** 2 / (len(X) - 1)
pca = PCA().fit(X)
print("SVD by hand   :", (variances / variances.sum())[:3].round(4))
print("sklearn PCA   :", pca.explained_variance_ratio_[:3].round(4))
print("components equal up to the sign:", np.allclose(np.abs(Vt), np.abs(pca.components_)))

The result is identical: `PCA` is an SVD of the centered data. The **sign** of the components is arbitrary (`v` and `-v` are the same direction): scikit-learn fixes one with a convention, other libraries another.
Never compare signs across different models.

## The scale decides the result

In the wine dataset the variance of `proline` is enormously larger than all the others. See what PCA does on the raw data and on the standardized ones.

In [ ]:
raw = PCA().fit(X)
scaled = PCA().fit(StandardScaler().fit_transform(X))
print("explained variance (raw data)         :", raw.explained_variance_ratio_[:4].round(3))
print("explained variance (standardized data):", scaled.explained_variance_ratio_[:4].round(3))
print("column with the largest loading in PC1 (raw):", X.columns[np.abs(raw.components_[0]).argmax()])

On the raw data the first component **is `proline`**: it explains 99.8% of the variance simply because that column has the largest variance. It is not a discovery about wines: it is an effect of the units of measure. After
standardization the variance spreads over more components and PCA discovers the true correlations between the measurements.

**Rule:** before a PCA you standardize, unless the columns already have the same unit and the same scale and you really want to preserve the differences in variance (pixels, homogeneous measurements).

> **Common mistake.** Interpreting the `explained_variance_ratio_` of a PCA on raw data as if it said "how important" a direction is. It says how much variance there is, which depends on the units.

## Projecting and reconstructing

`transform` projects the data onto the components (scores); `inverse_transform` brings them back to the original space. With all the components the reconstruction is exact; with `k` components it is the best possible **rank-`k` approximation**
(the closest in squared error).

In [ ]:
Z = StandardScaler().fit_transform(X)
for k in [1, 2, 5, 13]:
    p = PCA(k).fit(Z)
    error = ((Z - p.inverse_transform(p.transform(Z))) ** 2).mean()
    print(f"{k:2d} components | mean squared reconstruction error: {error:.4f} | unexplained variance: {1 - p.explained_variance_ratio_.sum():.4f}")

The mean reconstruction error coincides with the unexplained variance (for standardized data, where the total variance per column is 1): **the explained variance is a measure of reconstruction fidelity**, not of usefulness for predicting anything (you will see it in exercise C1).

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Why does PCA center the data?</summary>

Because variance is measured around the mean: without centering, the first direction would simply be the one going from the origin to the mean of the cloud. scikit-learn's `PCA` centers by itself (and saves the mean in `mean_`); `TruncatedSVD`, which does not center, is not the same thing (exercise E2).

</details>

<details><summary>Are the components the most important features of the dataset?</summary>

No. They are **linear combinations** of all the features (the weights are in `components_`, called loadings), ordered by variance and not by relationship with a target. PCA does not look at `y`: the most informative direction for predicting could be one of the last.

</details>